In [0]:
%pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib


In [1]:
SERVICE_ACCOUNT_FILE = "/dbfs/FileStore/keys/resume_dev.json"

In [ ]:
from google.oauth2 import service_account
from googleapiclient.discovery import build

FOLDER_ID = "your_folder_id_here"  # paste your folder ID here

creds = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE,
    scopes=["https://www.googleapis.com/auth/drive.readonly"]
)
service = build("drive", "v3", credentials=creds)

files = service.files().list(
    q=f"'{FOLDER_ID}' in parents and trashed=false",
    fields="files(id, name, mimeType)"
).execute().get("files", [])

for f in files:
    print(f['name'], "-", f['mimeType'])

In [ ]:
FOLDER_ID = "19WHz3NGVyd4c7r56dRPIXQeDYwu8o_Nn"

files = service.files().list(
    q=f"'{FOLDER_ID}' in parents and trashed=false",
    fields="files(id, name, mimeType)"
).execute().get("files", [])

for f in files:
    print(f['name'], "-", f['mimeType'])

In [ ]:
import io
import os
from googleapiclient.http import MediaIoBaseDownload

UC_VOLUME_PATH = "/Volumes/your_catalog/your_schema/your_volume"  # update this

EXPORT_MAP = {
    "application/vnd.google-apps.document":     ("application/vnd.openxmlformats-officedocument.wordprocessingml.document", ".docx"),
    "application/vnd.google-apps.spreadsheet":  ("application/vnd.openxmlformats-officedocument.spreadsheetml.sheet", ".xlsx"),
    "application/vnd.google-apps.presentation": ("application/vnd.openxmlformats-officedocument.presentationml.presentation", ".pptx"),
    "application/vnd.google-apps.drawing":      ("image/png", ".png"),
}

def download_folder(folder_id, dest_dir):
    os.makedirs(dest_dir, exist_ok=True)
    items = service.files().list(
        q=f"'{folder_id}' in parents and trashed=false",
        fields="files(id, name, mimeType)"
    ).execute().get("files", [])

    for f in items:
        file_id   = f["id"]
        file_name = f["name"]
        mime_type = f["mimeType"]

        if mime_type == "application/vnd.google-apps.folder":
            print(f"Entering subfolder: {file_name}")
            download_folder(file_id, f"{dest_dir}/{file_name}")

        elif mime_type in EXPORT_MAP:
            export_mime, ext = EXPORT_MAP[mime_type]
            request = service.files().export_media(fileId=file_id, mimeType=export_mime)
            if not file_name.endswith(ext):
                file_name += ext
            _save(request, f"{dest_dir}/{file_name}")

        elif mime_type.startswith("application/vnd.google-apps."):
            print(f"  Skipped (unsupported type): {file_name} [{mime_type}]")

        else:
            request = service.files().get_media(fileId=file_id)
            _save(request, f"{dest_dir}/{file_name}")

def _save(request, dest_path):
    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    buffer.seek(0)
    with open(dest_path, "wb") as out:
        out.write(buffer.read())
    print(f"  Copied → {dest_path}")

download_folder(FOLDER_ID, UC_VOLUME_PATH)